# Edge Vision — Optimization, Pruning & Hybrid Architecture

Selected model: **YOLO11n** (see Q3 scoring in `model_preview.ipynb`).

| Question | Where |
|---|---|
| Q1 baseline inference, Q2 lit review, Q3 model selection, Q4 demo | `model_preview.ipynb` / `report/` |
| Q5 optimization variants + LTH pruning | this notebook |
| Q6 benchmarking | this notebook + wandb |
| Q8 loss comparison, Q9 hybrid, Q10 second pruning technique | this notebook |

All runs are logged to the wandb project `edge-vision-yolo11n`.

Runtime: Colab A100. Note `onnxruntime-gpu` is deliberately **not** installed — it
replaces the cuDNN that PyTorch links against and breaks training.

## Setup

In [3]:
!pip install -q ultralytics wandb pycocotools     # no onnxruntime-gpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 151.9 MB/s eta 0:00:0000:01


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = "/content/drive/MyDrive/edge_vision"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

In [ ]:
!mkdir -p /content/datasets
!curl -sL -o /tmp/labels.zip https://github.com/ultralytics/yolov5/releases/download/v1.0/coco2017labels.zip
!unzip -q /tmp/labels.zip -d /content/datasets
!curl -sL -o /tmp/val2017.zip http://images.cocodataset.org/zips/val2017.zip
!unzip -q /tmp/val2017.zip -d /content/datasets/coco/images
!ls /content/datasets/coco && ls /content/datasets/coco/images/val2017 | head -3


In [2]:
import pathlib, random, yaml
import ultralytics
from ultralytics import settings

settings.update({"datasets_dir": "/content/datasets"})

paths = open("/content/datasets/coco/val2017.txt").read().split()
random.seed(0)                      # seeded, so the calibration set is reproducible
subset = random.sample(paths, 250)  # brief requires at least 200
open("/content/datasets/coco/calib250.txt", "w").write("\n".join(subset))

src = pathlib.Path(ultralytics.__file__).parent / "cfg/datasets/coco.yaml"
cfg = yaml.safe_load(src.read_text())
cfg["val"] = "calib250.txt"
yaml.safe_dump(cfg, open("/content/coco_calib.yaml", "w"))

print(len(subset), "calibration images")


250 calibration images


In [4]:
import wandb
wandb.login()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: trishab to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Q5 — Optimization variants

Baseline plus TensorRT FP32 / FP16 / INT8 and ONNX. `benchmark()` is the shared
export-and-measure pipeline every variant plugs into.

In [10]:
import os
import time

import numpy as np
import pandas as pd
import torch
import wandb
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetUtilizationRates
from ultralytics import YOLO

try:
    nvmlInit()
    gpu = nvmlDeviceGetHandleByIndex(0)
except Exception:
    gpu = None
    print("NVML unavailable - GPU utilisation will be logged as NaN")

SAMPLE = "/content/datasets/coco/images/val2017/000000000139.jpg"
ITERS = 100

def benchmark(name, weights, device=None):
    run = wandb.init(project="edge-vision-yolo11n", name=name, reinit=True)

    t0 = time.perf_counter()
    model = YOLO(weights)
    startup = time.perf_counter() - t0

    metrics = model.val(data="coco.yaml", device=device, verbose=False)

    for _ in range(10):                       # warm-up
        model(SAMPLE, device=device, verbose=False)

    latencies, gpu_util = [], []
    for _ in range(ITERS):
        start = time.perf_counter()
        model(SAMPLE, device=device, verbose=False)
        if device != "cpu":
            torch.cuda.synchronize()          # CUDA is async - time the work, not the queue
        latencies.append((time.perf_counter() - start) * 1000)
        if gpu is not None:
            gpu_util.append(nvmlDeviceGetUtilizationRates(gpu).gpu)

    latencies = np.array(latencies)
    stats = {
        "mAP50-95": metrics.box.map,
        "mAP50": metrics.box.map50,
        "latency_ms": latencies.mean(),
        "latency_std": latencies.std(),
        "fps": 1000 / latencies.mean(),
        "size_mb": os.path.getsize(weights) / 1e6,
        "gpu_util_pct": np.mean(gpu_util) if gpu_util else float("nan"),
        "startup_s": startup,
        "device": device or "cuda",
    }
    wandb.log(stats)
    stats["url"] = run.url    
    run.finish()
    print(name, {k: (round(v, 2) if isinstance(v, float) else v) for k, v in stats.items()})
    return stats


In [11]:
import shutil

results = {}

def export_engine(out_name, **kwargs):
    """Export from a fresh model each time - every engine export writes to
    yolo11n.engine, so they'd overwrite each other without renaming."""
    path = YOLO("yolo11n.pt").export(format="engine", **kwargs)
    shutil.move(path, out_name)
    return out_name

def run(name, weights, device=None):
    try:
        results[name] = benchmark(name, weights, device=device)
    except Exception as e:
        print(f"{name} FAILED: {type(e).__name__}: {e}")

run("yolo11n_baseline", "yolo11n.pt")

trt_fp32 = export_engine("yolo11n_fp32.engine")
run("yolo11n_tensorrt", trt_fp32)

trt_fp16 = export_engine("yolo11n_fp16.engine", half=True)
run("yolo11n_fp16", trt_fp16)

trt_int8 = export_engine("yolo11n_int8.engine", int8=True, data="/content/coco_calib.yaml")
run("yolo11n_int8", trt_int8)

onnx_path = YOLO("yolo11n.pt").export(format="onnx")
run("yolo11n_onnx_cpu", onnx_path, device="cpu")

pd.DataFrame(results).T


## Q5 — Lottery Ticket pruning

Three rounds of magnitude pruning, rewinding surviving weights to the pretrained
values, then retraining. COCO train2017 is 19 GB, so val2017 is split 4000/1000
and mAP is reported on the held-out 1000.

In [5]:
import pathlib, random, yaml
import ultralytics

paths = open("/content/datasets/coco/val2017.txt").read().split()
random.seed(0)
random.shuffle(paths)

open("/content/datasets/coco/lth_train.txt", "w").write("\n".join(paths[:4000]))
open("/content/datasets/coco/lth_eval.txt", "w").write("\n".join(paths[4000:]))

src = pathlib.Path(ultralytics.__file__).parent / "cfg/datasets/coco.yaml"
cfg = yaml.safe_load(src.read_text())
cfg["train"] = "lth_train.txt"
cfg["val"] = "lth_eval.txt"
yaml.safe_dump(cfg, open("/content/coco_lth.yaml", "w"))
print("4000 train / 1000 held-out eval")


4000 train / 1000 held-out eval


In [8]:
import copy
import torch
import wandb
from ultralytics import YOLO, settings

settings.update({"wandb": False})     # we name the runs ourselves

def conv_weights(net):
    convs = {n for n, m in net.named_modules() if isinstance(m, torch.nn.Conv2d)}
    return [(nm, p) for nm, p in net.named_parameters()
            if nm.endswith("weight") and nm.rsplit(".", 1)[0] in convs]

current = YOLO("yolo11n.pt")                              # weights used for magnitude ranking
theta0 = copy.deepcopy(current.model.state_dict())        # rewind target
lth = []

for rnd, amount in enumerate([0.2, 0.5, 0.7], start=1):
    # 1. rank by magnitude of the CURRENT trained weights
    pairs = conv_weights(current.model)
    allw = torch.cat([p.detach().abs().flatten() for _, p in pairs])
    thresh = torch.quantile(allw.float(), amount)
    masks = {nm: (p.detach().abs() > thresh).float().cpu() for nm, p in pairs}   # <- .cpu()

    # 2. rewind surviving weights to their original values, then apply the mask
    model = YOLO("yolo11n.pt")
    model.model.load_state_dict(theta0)
    with torch.no_grad():
        for nm, p in model.model.named_parameters():
            if nm in masks:
                p.mul_(masks[nm].to(p.device))  

    # 3. retrain, re-applying the mask after every step (model *and* EMA)
    def reapply(trainer, masks=masks):
        with torch.no_grad():
           for net in (trainer.model, getattr(getattr(trainer, "ema", None), "ema", None)):
                if net is None:
                    continue
                for nm, p in net.named_parameters():
                    if nm in masks:
                        if masks[nm].device != p.device:
                            masks[nm] = masks[nm].to(p.device)   # move once, then reuse
                        p.mul_(masks[nm])

    model.add_callback("on_train_batch_end", reapply)
    model.train(data="/content/coco_lth.yaml", epochs=3, imgsz=640, batch=32,
                verbose=False, plots=False)

    # 4. evaluate and log
    metrics = model.val(data="/content/coco_lth.yaml", verbose=False)
    zeros = sum((p == 0).sum().item() for nm, p in conv_weights(model.model))
    total = sum(p.numel() for nm, p in conv_weights(model.model))

    stats = {"round": rnd, "target_sparsity": amount,
             "actual_sparsity": zeros / total,
             "mAP50-95": metrics.box.map, "mAP50": metrics.box.map50}

    run = wandb.init(project="edge-vision-yolo11n", name=f"yolo11n_lth_round{rnd}", reinit=True)
    wandb.log(stats)
    run.finish()

    lth.append(stats)
    print(stats)
    current = model                                       # next round ranks from these weights


Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/coco_lth.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-5, nbs=64, nms=False,

actual_sparsity,▁
mAP50,▁
mAP50-95,▁
round,▁
target_sparsity,▁
actual_sparsity,0.20011
mAP50,0.52822
mAP50-95,0.37501
round,1
target_sparsity,0.2


{'round': 1, 'target_sparsity': 0.2, 'actual_sparsity': 0.20011472463021246, 'mAP50-95': np.float64(0.37501175648635265), 'mAP50': np.float64(0.5282195296810179)}
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/coco_lth.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr

actual_sparsity,▁
mAP50,▁
mAP50-95,▁
round,▁
target_sparsity,▁
actual_sparsity,0.5
mAP50,0.47067
mAP50-95,0.32712
round,2
target_sparsity,0.5


{'round': 2, 'target_sparsity': 0.5, 'actual_sparsity': 0.5, 'mAP50-95': np.float64(0.3271236435002769), 'mAP50': np.float64(0.4706657505846227)}
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/coco_lth.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

actual_sparsity,▁
mAP50,▁
mAP50-95,▁
round,▁
target_sparsity,▁
actual_sparsity,0.7
mAP50,0.27989
mAP50-95,0.18277
round,3
target_sparsity,0.7


{'round': 3, 'target_sparsity': 0.7, 'actual_sparsity': 0.700000076687587, 'mAP50-95': np.float64(0.18277127028116716), 'mAP50': np.float64(0.2798940885498546)}


## Q10 — Magnitude pruning without rewinding

Same protocol as the lottery-ticket runs above with exactly one change: each round
keeps the previous round's trained weights instead of rewinding to the originals.
Any difference between the two curves is therefore attributable to rewinding alone.

In [ ]:
base_metrics = YOLO("yolo11n.pt").val(data="/content/coco_lth.yaml", verbose=False)
baseline_map = base_metrics.box.map
print("unpruned mAP50-95 on held-out split:", round(baseline_map, 4))

In [ ]:
current = YOLO("yolo11n.pt")
magnitude = []

for rnd, amount in enumerate([0.2, 0.5, 0.7], start=1):
    pairs = conv_weights(current.model)
    allw = torch.cat([p.detach().abs().flatten() for _, p in pairs])
    thresh = torch.quantile(allw.float(), amount)
    masks = {nm: (p.detach().abs() > thresh).float().cpu() for nm, p in pairs}

    # no rewind - carry the trained weights forward
    prev_state = copy.deepcopy(current.model.state_dict())
    model = YOLO("yolo11n.pt")
    model.model.load_state_dict(prev_state)
    with torch.no_grad():
        for nm, p in model.model.named_parameters():
            if nm in masks:
                p.mul_(masks[nm].to(p.device))

    def reapply(trainer, masks=masks):
        with torch.no_grad():
            for net in (trainer.model, getattr(getattr(trainer, "ema", None), "ema", None)):
                if net is None:
                    continue
                for nm, p in net.named_parameters():
                    if nm in masks:
                        if masks[nm].device != p.device:
                            masks[nm] = masks[nm].to(p.device)
                        p.mul_(masks[nm])

    model.add_callback("on_train_batch_end", reapply)
    model.train(data="/content/coco_lth.yaml", epochs=3, imgsz=640, batch=32,
                verbose=False, plots=False)
    trained_ckpt = model.trainer.best
    metrics = model.val(data="/content/coco_lth.yaml", verbose=False)
    zeros = sum((p == 0).sum().item() for nm, p in conv_weights(model.model))
    total = sum(p.numel() for nm, p in conv_weights(model.model))

    stats = {"round": rnd, "target_sparsity": amount, "actual_sparsity": zeros / total,
             "mAP50-95": metrics.box.map, "mAP50": metrics.box.map50}

    run = wandb.init(project="edge-vision-yolo11n",
                     name=f"yolo11n_magnitude_round{rnd}", reinit=True)
    wandb.log(stats); run.finish()

    magnitude.append(stats)
    print(stats)
    current = YOLO(trained_ckpt)  

In [ ]:
import matplotlib.pyplot as plt

x_lth = [0] + [r["actual_sparsity"] * 100 for r in lth]
y_lth = [baseline_map] + [r["mAP50-95"] for r in lth]
x_mag = [0] + [r["actual_sparsity"] * 100 for r in magnitude]
y_mag = [baseline_map] + [r["mAP50-95"] for r in magnitude]

plt.figure(figsize=(7, 5))
plt.plot(x_lth, y_lth, "o-", label="Lottery ticket (prune + rewind + retrain)")
plt.plot(x_mag, y_mag, "s--", label="Magnitude pruning (fine-tune, no rewind)")
plt.xlabel("sparsity (%)")
plt.ylabel("mAP@0.5:0.95")
plt.title("YOLO11n: accuracy vs sparsity (1000-image held-out COCO split)")
plt.grid(alpha=0.3)
plt.legend()
plt.savefig(f"{OUT}/pruning_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Q9 — Hybrid architecture

YOLO11n with RT-DETR's AIFI encoder replacing YOLO11's own C2PSA attention block at P5.

In Q1, RT-DETR found all three traffic lights in the sample image and YOLO11n found
none. AIFI is the global-context self-attention that RT-DETR applies at P5, so the
question is whether it recovers that context at a fraction of RT-DETR's 20 M parameters.

Replacing C2PSA rather than inserting a new layer keeps every layer index stable, so
457/469 pretrained tensors still transfer.

In [ ]:
%%writefile yolo11n-aifi.yaml
# YOLO11n with RT-DETR's AIFI transformer encoder replacing C2PSA at P5
nc: 80
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]           # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]          # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]   # 2
  - [-1, 1, Conv, [256, 3, 2]]          # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]   # 4
  - [-1, 1, Conv, [512, 3, 2]]          # 5-P4/16
  - [-1, 2, C3k2, [512, True]]          # 6
  - [-1, 1, Conv, [1024, 3, 2]]         # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]         # 8
  - [-1, 1, SPPF, [1024, 5]]            # 9
  - [-1, 1, AIFI, [1024, 8]]            # 10  <-- RT-DETR encoder in place of C2PSA

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]                  # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]                  # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]                  # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]                  # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]]

In [ ]:
def train_and_eval(name, model, epochs=3):
    model.train(data="/content/coco_lth.yaml", epochs=epochs, imgsz=640, batch=32,
                verbose=False, plots=False)
    metrics = model.val(data="/content/coco_lth.yaml", verbose=False)
    stats = {"mAP50-95": metrics.box.map, "mAP50": metrics.box.map50,
             "params_m": sum(p.numel() for p in model.model.parameters()) / 1e6}
    run = wandb.init(project="edge-vision-yolo11n", name=name, reinit=True)
    wandb.log(stats); run.finish()
    print(name, stats)
    return stats


In [ ]:
# control uses the same data and epoch budget, so the comparison isolates the architecture
hybrid  = train_and_eval("yolo11n_aifi_hybrid", YOLO("yolo11n-aifi.yaml").load("yolo11n.pt"))
control = train_and_eval("yolo11n_control_finetune", YOLO("yolo11n.pt"))

## Q8 — Loss function comparison

Ultralytics uses `BCEWithLogitsLoss` for classification, not softmax cross-entropy,
so this compares BCE against focal-BCE.

Ultralytics ships a `FocalLoss`, but its forward returns a reduced scalar while
`v8DetectionLoss` needs per-element output of shape (batch, anchors, classes).
`FocalBCE` below applies the same focal modulation without reducing.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import ultralytics.utils.loss as ul

class FocalBCE(nn.Module):
    """Focal loss keeping per-element output, matching BCEWithLogitsLoss(reduction='none')."""

    def __init__(self, gamma=1.5, alpha=0.25):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha

    def forward(self, pred, label):
        loss = F.binary_cross_entropy_with_logits(pred, label, reduction="none")
        p = pred.sigmoid()
        p_t = label * p + (1 - label) * (1 - p)
        loss = loss * ((1.0 - p_t) ** self.gamma)
        if self.alpha > 0:
            loss = loss * (label * self.alpha + (1 - label) * (1 - self.alpha))
        return loss

_orig_init = ul.v8DetectionLoss.__init__

def focal_init(self, model, *args, **kwargs):
    _orig_init(self, model, *args, **kwargs)
    self.bce = FocalBCE()

ul.v8DetectionLoss.__init__ = focal_init
focal = train_and_eval("yolo11n_focal_loss", YOLO("yolo11n.pt"))

ul.v8DetectionLoss.__init__ = _orig_init      # restore before anything else trains
bce = train_and_eval("yolo11n_bce_loss", YOLO("yolo11n.pt"))

In [ ]:
import pandas as pd

pd.DataFrame([{"loss": "BCE (default)", **bce},
              {"loss": "Focal BCE", **focal}]).round(4)

## Save results

Run this after each question - a runtime disconnect then costs one question, not all of them.

In [ ]:
import json

def save():
    payload = {k: v for k, v in {
        "variants": globals().get("results"),
        "lth": globals().get("lth"),
        "magnitude": globals().get("magnitude"),
        "hybrid": globals().get("hybrid"),
        "control": globals().get("control"),
        "focal": globals().get("focal"),
        "bce": globals().get("bce"),
        "baseline_map": globals().get("baseline_map"),
    }.items() if v is not None}

    json.dump(payload, open(f"{OUT}/results.json", "w"), indent=2, default=str)
    for f in ["yolo11n-aifi.yaml", "yolo11n_fp16.engine", "yolo11n_int8.engine"]:
        if os.path.exists(f):
            shutil.copy(f, OUT)
    print("saved:", sorted(payload), "->", OUT)

save()